# 06 — Optional joint spatial/year transfer, chosen before final inspection

This is an optional SECOND experiment. Choose geographic holdouts from grid geometry **before** viewing final results. If chosen after inspecting residuals, label the analysis exploratory. CHT is small relative to radiometer footprints: a 35 km centre-distance buffer may leave too little fitting data. The notebook stops in that case rather than claiming independent spatial validation. It tests joint new-place/new-year transfer; it does not isolate a causal geography effect.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, json
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path("/content/cht-soil-moisture-gapfilling")]
             if (p / "pyproject.toml").exists()), None)
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/cht-soil-moisture-gapfilling")
    subprocess.run(["git", "clone", "--branch", "research-notebooks-v2", "--single-branch",
                    "https://github.com/Bipul-roy-AI/cht-soil-moisture-gapfilling.git", str(ROOT)], check=True)
if ROOT is None:
    raise RuntimeError("Open this notebook from the cloned repository, or run it in Colab.")
ROOT = ROOT.resolve()
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[gee,notebooks]"], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT)


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/CHT_SM_v2")
else:
    WORK = ROOT / "work_v2"
WORK.mkdir(parents=True, exist_ok=True)
print("Persistent work folder:", WORK)


In [ ]:
from chtsm.data import read_config, load_exports, audit_frame, save_json
CONFIG_PATH = WORK / "study_config.json"
config = read_config(CONFIG_PATH)
# Move the monthly FULL exports from the Drive export folder into this folder.
# Pilot exports have a different prefix and must not be mixed into the full table.
EXPORT_DIR = WORK / "exports"
CSV_PATHS = sorted(EXPORT_DIR.glob("chtsm_v2_full_*.csv"))
if not CSV_PATHS:
    raise FileNotFoundError("Put completed monthly chtsm_v2_full_*.csv exports in WORK/exports first.")
frame = load_exports(CSV_PATHS, config)


In [ ]:
from chtsm.gaps import spatial_subset
from chtsm.models import fit_experiment, freeze_experiment
import copy
cells = frame.groupby("cell_id")[["lon","lat"]].first()
display(cells)
HELDOUT_CELLS = []  # Explicit target-cell IDs selected using geographic geometry.
BUFFER_KM = 35
spatial_frame, design = spatial_subset(frame, HELDOUT_CELLS, buffer_km=BUFFER_KM)
spatial_config = copy.deepcopy(config);spatial_config["spatial_design"] = design
RUN = WORK / "runs" / "study_v2_spatial_001"
metadata, table = fit_experiment(spatial_frame, spatial_config, RUN, device="cpu")
display(table[["name", "validation_macro_cell_rmse"]])
print(design)

In [ ]:
FREEZE_SPATIAL_RUN = False
if FREEZE_SPATIAL_RUN:
    freeze_experiment(RUN, "Joint new-place/new-year transfer on predeclared held-out cells; 35 km approximate centre buffer; macro-cell RMSE primary; calibration-domain transfer assessed empirically.")
    save_json(WORK / "spatial_design.json", design)
else:
    print("Review retained spatial support and protocol, then explicitly enable freeze.")

For this experiment use `spatial_frame` and its spatial run path in the evaluation functions from notebook 05. Calibration uses fitting-region cells; interval coverage on held-out geography is not guaranteed. The production support policy will flag/withhold reconstruction in unseen cells even though raw transfer predictions are evaluated for research. If a useful holdout is impossible, report temporal generalization and narrow spatial claims rather than weakening the buffer silently.